In [ ]:
# =============================================================================
# 功能：将Excel文件中的两列数据（第一列为序号/键，第二列为JSON字符串）转换为
#       一个JSON对象文件。该脚本主要用于处理从其他系统导出的原始QA数据，
#       将每行的JSON字符串解析并存入字典，键为序号，值为解析后的JSON对象。
#       支持处理包含LaTeX命令等需要转义的特殊字符，提供容错机制，
#       输出格式化的JSON文件以供后续统计分析使用。
# =============================================================================
import pandas as pd
import json

# ================== 配置 ==================
EXCEL_PATH = "data/qa/2.unanswerable/200-unanswerable.xlsx"          # 输入 Excel 路径
OUTPUT_JSON_PATH = "data/qa/2.unanswerable/work__unanswerable__converted__batch01__n200.json"       # 输出 JSON 文件路径
SHEET_NAME = 0                         # 工作表索引或名称
# ==========================================

def safe_json_loads(s):
    """安全解析可能包含未转义反斜杠的 JSON 字符串"""
    if not isinstance(s, str):
        return None
    # 将单个反斜杠替换为双反斜杠（保留 LaTeX 命令）
    escaped = s.replace('\\', '\\\\')
    try:
        return json.loads(escaped)
    except json.JSONDecodeError as e:
        print(f"JSON 解析失败: {e}")
        return None

def main():
    # 读取 Excel（无表头，数据从第一行开始）
    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, header=None)
    
    if df.shape[1] < 2:
        print("错误：Excel 至少需要两列（第一列序号，第二列 JSON）")
        return

    result_dict = {}
    error_rows = []

    for idx in range(len(df)):
        row_num = idx + 1
        key = df.iloc[idx, 0]          # 第一列作为键
        json_str = df.iloc[idx, 1]     # 第二列 JSON 字符串

        # 跳过空行
        if pd.isna(key) or pd.isna(json_str):
            error_rows.append(row_num)
            continue

        # 将序号转为整数（如果原为浮点数）
        try:
            key_int = int(key)
        except:
            key_int = key  # 保持原样，但期望是数字

        # 解析 JSON
        parsed = safe_json_loads(json_str)
        if parsed is None:
            print(f"警告：第 {row_num} 行 JSON 解析失败，已跳过")
            error_rows.append(row_num)
            continue

        result_dict[key_int] = parsed

    # 写入 JSON 文件
    with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(result_dict, f, indent=2, ensure_ascii=False)

    print(f"成功转换 {len(result_dict)} 行")
    if error_rows:
        print(f"跳过或出错的行: {error_rows}")
    print(f"结果已保存至: {OUTPUT_JSON_PATH}")

if __name__ == "__main__":
    main()